In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def erosion_percentages(tif_path):
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata

    # remove NaNs
    if nodata is not None:
        data[data == nodata] = np.nan

    valid = data[~np.isnan(data)]

    total = valid.size

    # calculation of percent erosion and deposition
    neg = np.sum(valid < 0)
    pos = np.sum(valid > 0)
    zero = np.sum(valid == 0)

    return {
        "negative": 100 * neg / total,
        "zero": 100 * zero / total,
        "positive": 100 * pos / total,
    }

In [ ]:
# reading in files
files = {
    "2012–2013": "2013-2012.tif",
    "2013–2014": "2014-2013.tif",
    "2014–2015": "2015-2014.tif",
}

# running function on each file
results = {label: erosion_percentages(path) for label, path in files.items()}

In [ ]:
labels = list(results.keys())

# transformation of data required for plotting
neg_vals = [results[k]["negative"] for k in labels]
zero_vals = [results[k]["zero"] for k in labels]
pos_vals = [results[k]["positive"] for k in labels]

x = np.arange(len(labels))

# plotting
plt.figure(figsize=(6, 4))

plt.bar(x, neg_vals, label="Erosion", color="tab:red")
plt.bar(x, zero_vals, bottom=neg_vals, label="No Change", color="tab:gray")
plt.bar(x, pos_vals, bottom=np.array(neg_vals) + np.array(zero_vals), label="Deposition", color="tab:blue")

plt.xticks(x, labels)
plt.ylabel("Percentage of pixels (%)")
plt.title("Erosion vs. Deposition on Fire Island Post-Hurricane Sandy")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})
plt.legend()
plt.tight_layout()
plt.show()